## 1. Imports & Environment Setup
Importing deep learning frameworks, image processing tools, and visualization libraries.

In [ ]:
import os
import sys
import json
import time
import random
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

# Set styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version: {torch.__version__}')
print(f'Active Compute Device: {device}')

## 2. Dataset Paths & Class Distribution
Locate the dataset folders and inspect the 38 plant disease categories.

In [ ]:
# Determine local vs Kaggle paths
local_base = Path('plant/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)')
if not local_base.exists():
    local_base = Path('New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)')

train_dir = local_base / 'train'
valid_dir = local_base / 'valid'
test_dir = Path('plant/test/test') if Path('plant/test/test').exists() else Path('test/test')

classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(train_dir / d)])
num_classes = len(classes)

print(f'Total Classes: {num_classes}')
print(f'First 5 Classes: {classes[:5]}')

train_counts = {c: len(os.listdir(train_dir / c)) for c in classes}
valid_counts = {c: len(os.listdir(valid_dir / c)) for c in classes}

print(f'Total Training Images: {sum(train_counts.values()):,}')
print(f'Total Validation Images: {sum(valid_counts.values()):,}')

## 3. Exploratory Data Analysis & Visualizations

In [ ]:
# Plot Class Distribution
plt.figure(figsize=(18, 6))
bars = plt.bar(range(len(classes)), [train_counts[c] for c in classes], color='#10b981', edgecolor='black', alpha=0.8)
plt.xlabel('Disease Classes', fontsize=12, fontweight='bold')
plt.ylabel('Number of Images', fontsize=12, fontweight='bold')
plt.title('Training Data Distribution Across 38 Plant Disease Categories', fontsize=14, fontweight='bold')
plt.xticks(range(len(classes)), [c.replace('___', ' - ').replace('_', ' ') for c in classes], rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Healthy vs Diseased Ratio
healthy_count = sum(train_counts[c] for c in classes if 'healthy' in c.lower())
diseased_count = sum(train_counts[c] for c in classes if 'healthy' not in c.lower())

plt.figure(figsize=(6, 6))
plt.pie([healthy_count, diseased_count], labels=['Healthy Leaves', 'Infected / Diseased Leaves'], 
        autopct='%1.1f%%', colors=['#34d399', '#f87171'], explode=(0.05, 0), startangle=140)
plt.title('Healthy vs Diseased Samples in Dataset', fontweight='bold')
plt.show()

## 4. Data Augmentation & Custom Dataset Loaders

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PlantDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, target

print('Data transformations configured successfully.')

## 5. Transfer Learning Architecture (MobileNetV2)
We utilize a pretrained **MobileNetV2** backbone with fine-tuned top layers and a custom classification head.

In [ ]:
def build_model(num_classes):
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    
    # Freeze early convolutional layers
    for param in model.features.parameters():
        param.requires_grad = False
    # Unfreeze top feature layers
    for param in model.features[-3:].parameters():
        param.requires_grad = True
        
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes)
    )
    return model

model = build_model(num_classes).to(device)
print('MobileNetV2 Transfer Learning Model ready!')

## 6. Model Training & Evaluation History

In [ ]:
# Load training history if saved, or display results
history_path = Path('training_history.json')
if history_path.exists():
    with open(history_path, 'r') as f:
        hist = json.load(f)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs_range = range(1, len(hist['train_loss']) + 1)
    ax1.plot(epochs_range, hist['train_loss'], label='Training Loss', marker='o', color='#ef4444')
    ax1.plot(epochs_range, hist['val_loss'], label='Validation Loss', marker='s', color='#f59e0b')
    ax1.set_title('Cross Entropy Loss Curve', fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    
    ax2.plot(epochs_range, hist['train_acc'], label='Training Accuracy', marker='o', color='#10b981')
    ax2.plot(epochs_range, hist['val_acc'], label='Validation Accuracy', marker='s', color='#06b6d4')
    ax2.set_title('Classification Accuracy Curve (%)', fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print('Train the model using `python train_model.py` to generate training_history.json')

## 7. Unseen Test Image Inference & Diagnosis Preview

In [ ]:
# Inference Demonstration on Test Set
model_ckpt = Path('plant_disease_model.pth')
if model_ckpt.exists() and test_dir.exists():
    model.load_state_dict(torch.load(model_ckpt, map_location=device))
    model.eval()
    
    test_images = [test_dir / f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.png'))][:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    for ax, img_path in zip(axes.flat, test_images):
        img = Image.open(img_path).convert('RGB')
        inp = valid_transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(inp)
            prob = torch.softmax(out, dim=1)[0]
            top_p, top_i = torch.topk(prob, 1)
            pred_class = classes[top_i.item()]
            conf = top_p.item() * 100
            
        ax.imshow(img)
        ax.set_title(f'Pred: {pred_class}\nConf: {conf:.1f}%', fontsize=10, fontweight='bold',
                     color='green' if 'healthy' in pred_class.lower() else 'darkred')
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print('Checkpoint or test directory not found.')